In [0]:
%sql
create catalog if not exists realstate
managed location 'abfss://catalogs@datalakestorage000real.dfs.core.windows.net/'

In [0]:
%sql
create schema if not exists realstate.bronze;
create schema if not exists realstate.silver;
create schema if not exists realstate.gold;

# Posts table

In [0]:
%sql 
drop table realstate.silver.posts

In [0]:
%sql
create table IF NOT EXISTS realstate.silver.posts
(
  row_id INT,
    user_id BIGINT,
    user_name STRING,
    advertiser_type STRING,
    is_verified STRING,
    user_review FLOAT,
    city_id BIGINT,
    city STRING,
    district_id BIGINT,
    district STRING,
    property_type STRING,
    transaction_type STRING,
    location_lat DECIMAL(10,5),
    location_lng DECIMAL(10,5),
    create_time TIMESTAMP,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    age INT,
    has_air_condition STRING,
    has_kitchen STRING,
    bathrooms INT,
    bedrooms INT,
    livings INT,
    area DOUBLE,
    has_furnished STRING,
    price FLOAT,
    rent_period STRING,
    is_daily_rentable STRING
)
using delta
LOCATION 'abfss://silver@datalakestorage000real.dfs.core.windows.net/posts/'

## enable change feed 

In [0]:
%sql
ALTER TABLE realstate.silver.posts
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
-- drop table  realstate.silver.posts;
-- drop schema  realstate.silver;
-- drop schema  realstate.gold;
-- drop schema  realstate.bronze;
-- drop schema  realstate.default;
-- drop catalog realstate;
-- show schemas in realstate;
-- DESCRIBE DETAIL realstate.silver.posts

In [0]:
from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark, "realstate.silver.posts")
version=delta_table.history().select("version").collect()[0]["version"]
print(f" the version is {version}")
display(delta_table.history())


## metadata for ![](path)(change data feed)

In [0]:
%sql
create schema realstate.metadata
-- create table IF NOT EXISTS realstate

## for bronze layer

In [0]:
%sql
create table realstate.metadata.bronzeLayer(
    table_name string,
    last_ingestion string
) using delta
location 'abfss://metadata@datalakestorage000real.dfs.core.windows.net/bronze/'

In [0]:
%sql 
-- drop table realstate.metadata.bronzeLayer

## for gold

In [0]:
%sql
create table realstate.metadata.goldLayer(
    table_name string,
    last_version int
)using delta 
location 'abfss://metadata@datalakestorage000real.dfs.core.windows.net/gold/'

In [0]:
%sql
ALTER TABLE realstate.metadata.goldLayer
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql 
-- drop table realstate.metadata.goldLayer

In [0]:
%sql
create table realstate.metadata.goldLayer(
    table_name string,
    last_version int
)using delta
location 'abfss://metadata@datalakestorage000real.dfs.core.windows.net/gold/'

## set initial values

In [0]:
%sql
insert into realstate.metadata.bronzeLayer
values('comments','1970-01-01'),('posts','1970-01-01')

In [0]:
%sql
update realstate.metadata.bronzeLayer
set last_ingestion='1970-01-01'

In [0]:
%sql 
select * from realstate.metadata.bronzeLayer

In [0]:
%sql
describe history realstate.silver.posts;
-- describe detail realstate.metadata.silverLayer;

In [0]:
%sql
update realstate.metadata.goldLayer
set last_version=1

In [0]:
%sql
DESCRIBE HISTORY realstate.silver.posts

In [0]:
%sql
select * from realstate.gold.fact_posts where row_id=2929794  

In [0]:
%sql
select * from realstate.gold.dim_categories

In [0]:
%sql
select property_type,transaction_type from realstate.silver.posts